In [14]:
!pip install streamlit pyngrok -q
print("✅ Done")

✅ Done


In [22]:
!ngrok authtoken 2uX1TiKvIYnNtpjdv7LLl8Muk7g_4WYhbjTbrfVhHEbuDPEt3

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [23]:
%%writefile /content/app.py
import streamlit as st
import cv2
import numpy as np
from PIL import Image
import torch
import io
import os
import sys
import time

st.set_page_config(
    page_title="Open-Vocabulary Drone Analytics",
    page_icon="🚁",
    layout="wide"
)

st.sidebar.title("🚁 Drone Analytics")
st.sidebar.markdown("**Open-Vocabulary Detect → Segment**")
st.sidebar.markdown("---")
confidence = st.sidebar.slider("Detection Confidence", 0.1, 0.9, 0.25, 0.05)
use_sam2 = st.sidebar.checkbox("Enable SAM-2 Segmentation", value=False)
model_choice = st.sidebar.radio(
    "YOLO-World Model",
    ["Fine-tuned (VisDrone)", "Pretrained (General)"],
    index=0
)
st.sidebar.markdown("---")
st.sidebar.markdown("Detects any object described in natural language.")

DRIVE_ROOT = "/content/drive/MyDrive/DLCV_OV_Analytics"
YW_FT_CKPT = f"{DRIVE_ROOT}/checkpoints/yoloworld/yolov8l-world-finetuned.pt"
YW_PT_CKPT = f"{DRIVE_ROOT}/checkpoints/yoloworld/yolov8l-world.pt"
SAM2_CKPT  = f"{DRIVE_ROOT}/checkpoints/sam2/sam2.1_hiera_large.pt"
SAM2_REPO  = "/content/sam2"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

COLORS = [
    (255,80,80),  (80,120,255), (80,180,80),  (255,160,80),
    (180,80,255), (255,220,60), (80,220,220),  (200,120,80),
    (255,100,180),(100,200,100),(180,180,180), (60,180,255),
]

@st.cache_resource
def load_yolo(model_path):
    original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return original_load(*args, **kwargs)
    torch.load = patched_load
    from ultralytics import YOLO
    model = YOLO(model_path)
    model.to(DEVICE)
    torch.load = original_load
    return model

@st.cache_resource
def load_sam2_predictor(sam2_repo, ckpt_path):
    if sam2_repo not in sys.path:
        sys.path.insert(0, sam2_repo)
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    model = build_sam2(
        "configs/sam2.1/sam2.1_hiera_l.yaml",
        ckpt_path, device=DEVICE
    )
    return SAM2ImagePredictor(model)

st.title("🚁 Open-Vocabulary Drone-View Analytics")
st.markdown("**Type what you want to detect. Upload an image. Get results.**")
st.markdown("---")

col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("1. Enter your prompt")
    prompt_input = st.text_area(
        "Object categories (comma-separated)",
        value="pedestrian, car, bus, truck, motorcycle",
        height=80
    )
    st.caption("💡 Try: 'moving vehicle, parked car' or 'person walking, group of people'")
    st.subheader("2. Upload image")
    uploaded_file = st.file_uploader(
        "Upload a drone-view image (JPG/PNG)",
        type=["jpg", "jpeg", "png"]
    )
    run_button = st.button("🔍 Detect & Segment", type="primary",
                            use_container_width=True)

with col2:
    st.subheader("Results")
    result_placeholder = st.empty()
    metrics_placeholder = st.empty()

if run_button and uploaded_file is not None:
    classes = [c.strip() for c in prompt_input.split(",") if c.strip()]
    if not classes:
        st.error("Please enter at least one class.")
        st.stop()

    img_np = np.array(Image.open(uploaded_file).convert("RGB"))

    with st.spinner("Loading model..."):
        ckpt  = YW_FT_CKPT if model_choice == "Fine-tuned (VisDrone)" else YW_PT_CKPT
        model = load_yolo(ckpt)
        model.set_classes(classes)

    with st.spinner(f"Detecting: {', '.join(classes[:3])}..."):
        t0 = time.time()
        results = model.predict(img_np, verbose=False, conf=confidence)
        det_time = time.time() - t0

    detections = []
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            cid = int(box.cls.item())
            detections.append({
                "class": classes[cid] if cid < len(classes) else str(cid),
                "conf" : float(box.conf.item()),
                "box"  : box.xyxy[0].tolist(),
            })

    canvas  = img_np.copy()
    seg_time = 0

    if use_sam2 and detections and os.path.exists(SAM2_CKPT):
        with st.spinner("Segmenting with SAM-2..."):
            try:
                predictor = load_sam2_predictor(SAM2_REPO, SAM2_CKPT)
                predictor.set_image(img_np)
                overlay = canvas.copy().astype(float)
                t0 = time.time()
                for i, det in enumerate(detections):
                    color  = np.array(COLORS[i % len(COLORS)], dtype=float)
                    box_np = np.array([det["box"]], dtype=np.float32)
                    try:
                        masks, _, _ = predictor.predict(
                            box=box_np, multimask_output=False
                        )
                        if masks is not None and len(masks) > 0:
                            mask = masks[0].squeeze().astype(bool)
                            overlay[mask] = overlay[mask]*0.4 + color*0.6
                    except Exception:
                        pass
                canvas   = (overlay*0.7 + canvas.astype(float)*0.3).astype(np.uint8)
                seg_time = time.time() - t0
            except Exception as e:
                st.warning(f"SAM-2 failed: {e}. Showing detections only.")

    for i, det in enumerate(detections):
        color = COLORS[i % len(COLORS)]
        x1,y1,x2,y2 = [int(v) for v in det["box"]]
        cv2.rectangle(canvas, (x1,y1), (x2,y2), color, 2)
        label = f"{det['class']} {det['conf']:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(canvas, (x1, max(0,y1-th-6)), (x1+tw+4, y1), color, -1)
        cv2.putText(canvas, label, (x1+2, max(0,y1-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

    with col2:
        result_placeholder.image(canvas, caption="Detection Result",
                                  use_container_width=True)
        class_counts = {}
        for d in detections:
            class_counts[d["class"]] = class_counts.get(d["class"], 0) + 1
        metrics_placeholder.markdown(
            f"**🎯 Objects found:** {len(detections)}  \n"
            f"**⚡ Detection time:** {det_time*1000:.0f} ms  \n"
            f"**🎭 Segmentation time:** {seg_time*1000:.0f} ms  \n"
            f"**📝 Prompt:** `{', '.join(classes)}`  \n\n" +
            "\n".join([f"- {cls}: {cnt}" for cls, cnt in class_counts.items()])
        )

    buf = io.BytesIO()
    Image.fromarray(canvas).save(buf, format="PNG")
    st.download_button("📥 Download Result", buf.getvalue(),
                       file_name="result.png", mime="image/png")

elif run_button and uploaded_file is None:
    with col2:
        result_placeholder.warning("Please upload an image first.")

if not run_button:
    with col2:
        result_placeholder.info("👈 Enter a prompt, upload an image, then click Detect & Segment.")
        st.markdown("""
### Example prompts:
- `pedestrian, car, bus` — VisDrone categories
- `moving vehicle, parked car` — descriptive prompts
- `person walking, group of people` — person-focused
- `large vehicle, small vehicle` — size-based queries
        """)

Overwriting /content/app.py


In [17]:
from google.colab import drive
drive.mount('/content/drive')

# Verify the checkpoint exists
import os
ckpt = "/content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/yoloworld/yolov8l-world-finetuned.pt"
print("Found ✅" if os.path.exists(ckpt) else "NOT FOUND ❌")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found ✅


In [19]:
import subprocess, time
from pyngrok import ngrok

subprocess.run("pkill -9 -f streamlit", shell=True)
ngrok.kill()
time.sleep(4)

subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.address", "0.0.0.0"
])

time.sleep(10)
tunnel = ngrok.connect(8501, "http")
print(f"URL: {tunnel.public_url}")

URL: https://1411-34-125-206-197.ngrok-free.app


In [20]:
import subprocess, time
from pyngrok import ngrok

subprocess.run("pkill -9 -f streamlit", shell=True)
ngrok.kill()
time.sleep(3)

# Start streamlit and capture output
proc = subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.address", "0.0.0.0"
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

time.sleep(8)

# Check if it started
out, err = proc.communicate(timeout=2) if proc.poll() is not None else (b"", b"")
print("STDOUT:", out.decode()[:500] if out else "running")
print("STDERR:", err.decode()[:500] if err else "no errors")

# Check port
result = subprocess.run("lsof -i :8501", shell=True, capture_output=True, text=True)
print("Port 8501:", "LISTENING ✅" if result.stdout.strip() else "NOT LISTENING ❌")

if result.stdout.strip():
    tunnel = ngrok.connect(8501, "http")
    print(f"URL: {tunnel.public_url}")
else:
    print("Streamlit failed to start — check stderr above")

STDOUT: running
STDERR: no errors
Port 8501: LISTENING ✅
URL: https://d33b-34-125-206-197.ngrok-free.app


In [24]:
import subprocess, shutil, os, time
from pyngrok import ngrok

# Step 1 — Copy checkpoint to local fast storage
print("Copying checkpoint to local storage...")
os.makedirs("/content/checkpoints", exist_ok=True)

shutil.copy(
    "/content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/yoloworld/yolov8l-world-finetuned.pt",
    "/content/checkpoints/yolov8l-world-finetuned.pt"
)
shutil.copy(
    "/content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/yoloworld/yolov8l-world.pt",
    "/content/checkpoints/yolov8l-world.pt"
)
print("✅ Checkpoints copied to local storage")

# Step 2 — Update paths in app.py
with open("/content/app.py", "r") as f:
    content = f.read()

content = content.replace(
    'YW_FT_CKPT = f"{DRIVE_ROOT}/checkpoints/yoloworld/yolov8l-world-finetuned.pt"',
    'YW_FT_CKPT = "/content/checkpoints/yolov8l-world-finetuned.pt"'
)
content = content.replace(
    'YW_PT_CKPT = f"{DRIVE_ROOT}/checkpoints/yoloworld/yolov8l-world.pt"',
    'YW_PT_CKPT = "/content/checkpoints/yolov8l-world.pt"'
)

with open("/content/app.py", "w") as f:
    f.write(content)
print("✅ Paths updated to local storage")

# Step 3 — Restart
subprocess.run("pkill -9 -f streamlit", shell=True)
ngrok.kill()
time.sleep(4)

subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.address", "0.0.0.0"
])

time.sleep(10)
tunnel = ngrok.connect(8501, "http")
print(f"\n🚀 URL: {tunnel.public_url}")

Copying checkpoint to local storage...
✅ Checkpoints copied to local storage
✅ Paths updated to local storage

🚀 URL: https://905b-34-125-206-197.ngrok-free.app
